<a href="https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis**: One row = one `content_hash_id`, for one `client_hash_id`, on one report_date — a content × day grain in `fact_content_daily_performance`, joined to `dim_content` (static content attributes) on `content_hash_id`.

**Table(s)**: `fact_content_daily_performance` (daily metrics: impressions, clicks, sessions, AI-referral counts) joined to `dim_content` (static: `content_type`, `word_count`, `main_intent`, `provider_used`, etc.) via `content_hash_id`.

**Time window**: September 2025 — a full calendar month, chosen as a genuine mid-panel month. The full panel spans 18 months (2025-01 through 2026-06, confirmed via list_repo_files against the month-partitioned parquet structure, not by streaming), so September sits near the true midpoint — clear of both the panel's start (no prior history to build features from) and its end (no future window left to construct a label from). Loaded directly via DuckDB against the month=2025-09 partition, giving 845,813 rows — the full month, no sampling needed.

**What I'd predict/rank**: No pre-built label exists in this table. The proxy label is constructed from `gsc_impressions`: sum it over a prior sub-window vs. a later sub-window within September, compute % change, then bucket into up/down/stable.

Deliberately excluded: All hash IDs (`content_hash_id`, `client_hash_id`, `keyword_hash_id`, `url_hash_id`) — pure identifiers, no predictive signal. Also excluded from the feature set (but used for filtering): `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` — availability flags, not content signals.

In [ ]:

!git clone https://github.com/Hadeed07/FlyRank-ML.git
import pandas as pd

fatal: destination path 'FlyRank-ML' already exists and is not an empty directory.


In [ ]:
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
ds2 = load_dataset("FlyRank/internship-warehouse", "dim_content", streaming=True, split="train")

In [ ]:
sample = list(ds.take(3))
sample[0]

{'report_date': datetime.date(2025, 1, 27),
 'client_hash_id': 'client_9958f0a7ae1df715',
 'content_hash_id': 'content_3b70a18ea133b2bb',
 'client_has_gsc': True,
 'client_has_ga4': True,
 'gsc_data_available': True,
 'ga4_data_available': False,
 'gsc_impressions': 30,
 'gsc_clicks': 0,
 'gsc_sum_position': 115,
 'gsc_avg_position': 3.8333333333333335,
 'ga4_pageviews': 0,
 'ga4_sessions': 0,
 'ga4_users': 0,
 'ga4_engaged_sessions': 0,
 'ga4_total_engagement_sec': 0,
 'sessions_organic': 0,
 'sessions_direct': 0,
 'sessions_referral': 0,
 'sessions_social': 0,
 'sessions_paid': 0,
 'sessions_ai': 0,
 'ai_chatgpt': 0,
 'ai_perplexity': 0,
 'ai_gemini': 0,
 'ai_copilot': 0,
 'ai_claude': 0,
 'ai_meta': 0,
 'ai_other': 0,
 'scroll_events': 0}

In [ ]:
sample = list(ds2.take(3))
sample[0]

{'client_hash_id': 'client_04660893ae39614a',
 'content_hash_id': 'content_004de9653278b5a4',
 'keyword_hash_id': 'keyword_e754999ab88dd9f2',
 'url_hash_id': 'url_d6091f18cf628794',
 'keyword_char_count': 22,
 'keyword_token_count': 4,
 'url_char_count': 108,
 'content_created_date': datetime.date(2026, 5, 30),
 'content_updated_date': datetime.date(2026, 7, 1),
 'content_type': 'keyword article',
 'search_volume': 30,
 'competition': 0.91,
 'competition_level': 'HIGH',
 'cpc': 0.98,
 'main_intent': 'transactional',
 'backlinks': 16,
 'category_count': 3,
 'keyword_created_date': datetime.date(2026, 5, 12),
 'provider_used': 'gemini-generate-content',
 'model_used': 'gemini-3-flash-preview',
 'char_count': 15682,
 'word_count': 2555,
 'last_optimized_date': None,
 'optimization_eligible_date': None,
 'is_published': True,
 'is_deleted': False}

In [ ]:
import itertools
rows = list(itertools.islice(ds, 5000))
dates = [r['report_date'] for r in rows]
print(min(dates), max(dates))

In [ ]:
def in_feb_2025(row):
    return row['report_date'].year == 2025 and row['report_date'].month == 2

feb_ds = ds.filter(in_feb_2025)
feb_rows = list(itertools.islice(feb_ds, 20_000))  # cap so it doesn't run forever
import pandas as pd
feb_df = pd.DataFrame(feb_rows)
feb_df.shape

Checking the duplicates: Grain Check

In [ ]:
print("Total rows:", len(feb_df))
print("Unique (content_hash_id, report_date) pairs:", feb_df.drop_duplicates(['content_hash_id','report_date']).shape[0])

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


Give below are the columns for features/ labels / context / exclude.


`days_with_impressions`, `days_with_sessions` are the columns with no window suffix. These columns could span full 90 days windows or just one sub-window. So we don't trust them.

`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier` — this is existing unresolved group, unsuffixed ratios/tiers of unconfirmed source window.

`trend_pct` is almost certainly the raw percentage change between the prior and later window that trend_direction was then bucketed from (e.g., "-15% → down", "+8% → up"). This is potential for data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **No pre-built label exists.** Unlike the old starter CSV's `trend_direction`, this table
  has no ground-truth trend label. `trend_direction` here is a proxy constructed from
  `gsc_impressions` (prior 15 days vs. later 15 days, % change, bucketed). It reflects a
  modeling choice, not a verified outcome — a different metric (e.g. `gsc_clicks`) or a
  different split ratio could produce a different label for the same content.

- **Single-month slice, not the full panel.** The full dataset spans 18 months
  (`2025-01` through `2026-06`, confirmed via the month-partitioned file listing).
  September 2025 was chosen as a genuine mid-panel month, clear of both the panel's
  start (no prior history) and end (no future window). But findings from this one month
  — availability rates, date-field behavior, client counts — are not guaranteed to hold
  in other months, especially given likely seasonal effects on search/content behavior.

- **Availability flags showed zero variance in this slice.** `gsc_data_available` and
  `client_has_gsc` were both `True` for all 845,813 rows — confirmed via cross-tab. This
  means the September slice happens to contain no non-GSC clients; the filtering already
  happened upstream, not row-by-row within this table. A different month or the full
  panel could show real variation, so this can't be assumed to generalize.

- **Content lifecycle fields cluster heavily after the decision cutoff.** Three derived
  date-based features (`days_since_update`, `days_since_last_optimized`,
  `is_optimization_eligible_yet`) were tested with leakage-safe gating at the Sept 15
  cutoff and all failed: 99.5% of update events and 100% of optimization events occurred
  *after* the cutoff (or never occurred at all) in this slice. This means the dataset
  can't currently answer "how fresh was this content as of the decision point" — a gap,
  not just a dropped feature.

- **Observed field redundancy, not investigated further.** `last_optimized_date` and
  `optimization_eligible_date` produced identical leak/never-occurred counts (14,812 /
  38,315), suggesting they may derive from the same underlying event rather than being
  independent. Flagged, not confirmed.

- **Client concentration not yet checked for this slice.** The old CSV showed 2 of 32
  clients holding ~33% of rows combined, motivating the grouped-split requirement. This
  slice has 23 unique clients — whether a similar concentration exists here is unverified
  and should be checked before the train/test split is built.

- **Anonymization artifact, unresolved.** An earlier `dim_content` sample showed a
  `content_created_date` of 2026-05-30 — later than any date in

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.